# First 30

In [ ]:
import pandas as pd

R_SET_PATH = "/content/R-set.csv"
T_SET_PATH = "/content/T-set.csv"

OUTPUT_DIR = "/content/external_requirement_benchmark_inputs"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

r_df = pd.read_csv(R_SET_PATH, sep=";", encoding="utf-8-sig", engine="python")
t_df = pd.read_csv(T_SET_PATH, sep=";", encoding="utf-8-sig", engine="python")

r30 = r_df.head(30).copy()
t30 = t_df.head(30).copy()

r30.to_csv(os.path.join(OUTPUT_DIR, "R-set_first_30.csv"), index=False, sep=";")
t30.to_csv(os.path.join(OUTPUT_DIR, "T-set_first_30.csv"), index=False, sep=";")

print("Saved:")
print(os.path.join(OUTPUT_DIR, "R-set_first_30.csv"))
print(os.path.join(OUTPUT_DIR, "T-set_first_30.csv"))

Saved:
/content/external_requirement_benchmark_inputs/R-set_first_30.csv
/content/external_requirement_benchmark_inputs/T-set_first_30.csv


In [ ]:
# ============================================================
# External Requirement-to-Test Benchmark
# Intent-level semantic matching
#
# Input:
#   R-set.csv              -> first 30 requirements
#   T-set.csv              -> first 30 reference test cases
#   Geneated_testcases.csv -> generated test cases from one model
#
# Main idea:
#   Different formats are converted into a common "test intent" text.
#   Then BGE-large + cosine similarity is used for semantic matching.
#
# Outputs:
#   1. summary metrics
#   2. reference best matches
#   3. generated best matches
#   4. strict one-to-one matches
# ============================================================

import os
import re
import shutil
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

# =========================
# 1. FILE PATHS
# =========================

# For Colab, change these paths to your Google Drive paths.
R_SET_PATH = "/content/external_requirement_benchmark_inputs/R-set_first_30.csv"
T_SET_PATH = "/content/external_requirement_benchmark_inputs/T-set_first_30.csv"
GENERATED_PATH = "/content/Geneated_testcases.csv"

OUTPUT_DIR = "/content/external_requirement_intent_matching_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_LABEL = "Qwen-7B"   # change this for Qwen-14B, Mistral-7B, Mistral-14B

MODEL_NAME = "BAAI/bge-large-en-v1.5"
CACHE_DIR = "/content/bge_large_cache"

THRESHOLDS = [0.65, 0.70, 0.75]
PRIMARY_THRESHOLD = 0.70

USE_BGE_INSTRUCTION = True
BGE_INSTRUCTION = "Represent this software testing intent for semantic similarity matching: "

# Keep False if BGE-large is already downloaded.
CLEAN_LOCAL_MODEL_CACHE = False

# =========================
# 2. LOAD BGE-LARGE
# =========================

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

try:
    import torch
except Exception:
    torch = None

from sentence_transformers import SentenceTransformer

if CLEAN_LOCAL_MODEL_CACHE and os.path.exists(CACHE_DIR):
    print("Removing cache:", CACHE_DIR)
    shutil.rmtree(CACHE_DIR, ignore_errors=True)

os.makedirs(CACHE_DIR, exist_ok=True)

device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Loading model:", MODEL_NAME)

try:
    model = SentenceTransformer(
        MODEL_NAME,
        device=device,
        cache_folder=CACHE_DIR,
        model_kwargs={"use_safetensors": False}
    )
    print("Loaded BGE-large using pytorch_model.bin route.")
except TypeError:
    model = SentenceTransformer(
        MODEL_NAME,
        device=device,
        cache_folder=CACHE_DIR
    )
    print("Loaded BGE-large using standard route.")

print("Model ready:", MODEL_NAME)

# =========================
# 3. BASIC HELPERS
# =========================

def normalize_text(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = text.lower()
    text = text.replace("\n", " ")
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def threshold_key(t):
    return str(t).replace(".", "_")


def prepare_for_embedding(texts):
    cleaned = [normalize_text(t) for t in texts]
    if USE_BGE_INSTRUCTION:
        cleaned = [BGE_INSTRUCTION + t for t in cleaned]
    return cleaned


def encode_texts(texts, batch_size=16):
    return model.encode(
        prepare_for_embedding(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )


def get_first_existing_value(row, candidates):
    for c in candidates:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c]).strip()
    return ""


def read_semicolon_csv(path):
    return pd.read_csv(
        path,
        sep=";",
        encoding="utf-8-sig",
        engine="python"
    )


# =========================
# 4. REFERENCE TEST CASE PARSING
# =========================

def extract_section(text, start_keywords, end_keywords):
    """
    Extracts sections from long T-set reference scripts.
    Example sections:
      INITIAL CONDITIONS
      STEPS/DESCRIPTION
      EXPECTED RESULTS
      LINKS
    """
    raw = "" if text is None or pd.isna(text) else str(text)

    upper = raw.upper()
    start_pos = None

    for key in start_keywords:
        pos = upper.find(key.upper())
        if pos != -1:
            start_pos = pos + len(key)
            break

    if start_pos is None:
        return ""

    end_pos = len(raw)

    for end_key in end_keywords:
        pos = upper.find(end_key.upper(), start_pos)
        if pos != -1:
            end_pos = min(end_pos, pos)

    return raw[start_pos:end_pos].strip()


def remove_numbering(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = re.sub(r"\b\d+\.\s*", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_reference_intent(row):
    """
    Converts R-set + T-set reference script into a compact test-intent representation.

    This is important because T-set reference cases are long procedural scripts,
    while generated cases are short structured cases.
    """
    req_id = str(row.get("RequirementID", ""))
    requirement = str(row.get("RequirementContent", ""))
    test_case_id = str(row.get("TestCaseID", ""))
    test_content = str(row.get("TestCaseContent", ""))

    initial_conditions = extract_section(
        test_content,
        start_keywords=["INITIAL CONDITIONS"],
        end_keywords=["STEPS/DESCRIPTION", "EXPECTED RESULTS", "LINKS"]
    )

    steps = extract_section(
        test_content,
        start_keywords=["STEPS/DESCRIPTION", "STEPS"],
        end_keywords=["EXPECTED RESULTS", "LINKS"]
    )

    expected = extract_section(
        test_content,
        start_keywords=["EXPECTED RESULTS", "EXPECTED RESULT"],
        end_keywords=["LINKS"]
    )

    # The intent text gives higher importance to requirement + expected behavior.
    # Full T-set content is also included lightly so details are not lost.
    intent = (
        f"Requirement ID: {req_id}. "
        f"Requirement intent: {requirement}. "
        f"Reference test case ID: {test_case_id}. "
        f"Setup context: {remove_numbering(initial_conditions)}. "
        f"User action or test steps: {remove_numbering(steps)}. "
        f"Expected behavior: {remove_numbering(expected)}."
    )

    return intent


# =========================
# 5. GENERATED TEST CASE PARSING
# =========================

def build_generated_intent(row):
    """
    Converts any generated test-case format into the same compact intent representation.

    This works even if the generated CSV columns do not exactly match the T-set format.
    """
    generated_id = get_first_existing_value(
        row,
        ["Test Case ID", "test_case_id", "testcase_id", "ID", "id"]
    )

    title = get_first_existing_value(
        row,
        ["Title", "title", "Test Case Title", "test_title", "Name", "name"]
    )

    module = get_first_existing_value(
        row,
        ["Module", "module", "Feature", "feature", "Component", "component"]
    )

    requirement_id = get_first_existing_value(
        row,
        [
            "Requirement ID", "RequirementID", "requirement_id",
            "Source Requirement", "source_requirement", "Requirement"
        ]
    )

    preconditions = get_first_existing_value(
        row,
        ["Preconditions", "preconditions", "Precondition", "precondition"]
    )

    test_data = get_first_existing_value(
        row,
        ["Test Data", "test_data", "Data", "data"]
    )

    steps = get_first_existing_value(
        row,
        ["Test Steps", "test_steps", "Steps", "steps", "Procedure", "procedure"]
    )

    expected = get_first_existing_value(
        row,
        ["Expected Result", "expected_result", "Expected Results", "expected_results", "Oracle", "oracle"]
    )

    # Fallback: include all columns so no information is lost if the model output format changes.
    all_columns_text = " ".join(
        [str(x) for x in row.values if pd.notna(x) and str(x).strip()]
    )

    intent = (
        f"Generated test case ID: {generated_id}. "
        f"Test objective title: {title}. "
        f"Feature or module: {module}. "
        f"Requirement reference: {requirement_id}. "
        f"Precondition context: {preconditions}. "
        f"Input or test data: {test_data}. "
        f"User action or test steps: {remove_numbering(steps)}. "
        f"Expected behavior: {remove_numbering(expected)}. "
        f"Full generated case: {all_columns_text}."
    )

    return intent


def read_generated_cases(path):
    # The generated cases file `Geneated_testcases.csv` is likely comma-separated.
    # The `read_semicolon_csv` function was causing a ParserError.
    df = pd.read_csv(path).copy().reset_index(drop=True)

    if "Test Case ID" in df.columns:
        df["generated_id"] = df["Test Case ID"].astype(str)
    elif "test_case_id" in df.columns:
        df["generated_id"] = df["test_case_id"].astype(str)
    elif "testcase_id" in df.columns:
        df["generated_id"] = df["testcase_id"].astype(str)
    elif "ID" in df.columns:
        df["generated_id"] = df["ID"].astype(str)
    else:
        df["generated_id"] = [f"GEN-{i+1:03d}" for i in range(len(df))]

    df["generated_intent_text"] = df.apply(build_generated_intent, axis=1)
    df["generated_intent_norm"] = df["generated_intent_text"].apply(normalize_text)

    return df


# =========================
# 6. LOAD R-SET, T-SET, GENERATED CASES
# =========================

r_df = read_semicolon_csv(R_SET_PATH)
t_df = read_semicolon_csv(T_SET_PATH)

# First 30 requirement-test pairs.
r30 = r_df.head(30).copy().reset_index(drop=True)
t30 = t_df.head(30).copy().reset_index(drop=True)

reference_df = pd.concat(
    [
        r30[["RequirementID", "RequirementContent"]],
        t30[["TestCaseID", "TestCaseContent"]]
    ],
    axis=1
)

reference_df["reference_intent_text"] = reference_df.apply(build_reference_intent, axis=1)
reference_df["reference_intent_norm"] = reference_df["reference_intent_text"].apply(normalize_text)

generated_df = read_generated_cases(GENERATED_PATH)

print("Reference cases:", len(reference_df))
print("Generated cases:", len(generated_df))

# Save the normalized intent representations for inspection.
reference_intent_path = os.path.join(OUTPUT_DIR, "reference_intent_texts.csv")
generated_intent_path = os.path.join(OUTPUT_DIR, "generated_intent_texts.csv")

reference_df.to_csv(reference_intent_path, index=False)
generated_df.to_csv(generated_intent_path, index=False)

print("Saved intent text files:")
print(reference_intent_path)
print(generated_intent_path)

# =========================
# 7. INTENT-LEVEL SEMANTIC MATCHING
# =========================

ref_emb = encode_texts(reference_df["reference_intent_norm"].tolist())
gen_emb = encode_texts(generated_df["generated_intent_norm"].tolist())

sim_matrix = cosine_similarity(ref_emb, gen_emb)

# =========================
# 8. REFERENCE RECALL SIDE
# =========================
# For each reference test case:
# Which generated test case best recovers this reference intent?

ref_best_idx = sim_matrix.argmax(axis=1)
ref_best_score = sim_matrix.max(axis=1)

reference_match_rows = []

for i, ref_row in reference_df.iterrows():
    best_j = int(ref_best_idx[i])

    row = {
        "model": MODEL_LABEL,
        "RequirementID": ref_row["RequirementID"],
        "TestCaseID": ref_row["TestCaseID"],
        "RequirementContent": ref_row["RequirementContent"],
        "ReferenceTestCaseContent": ref_row["TestCaseContent"],
        "reference_intent_text": ref_row["reference_intent_text"],
        "best_generated_id": generated_df.iloc[best_j]["generated_id"],
        "best_generated_intent_text": generated_df.iloc[best_j]["generated_intent_text"],
        "best_similarity": float(ref_best_score[i])
    }

    for tau in THRESHOLDS:
        row[f"reference_matched_at_{threshold_key(tau)}"] = int(ref_best_score[i] >= tau)

    reference_match_rows.append(row)

reference_matches_df = pd.DataFrame(reference_match_rows)

# =========================
# 9. GENERATED PRECISION SIDE
# =========================
# For each generated test case:
# Which reference test case does it best correspond to?

gen_best_idx = sim_matrix.argmax(axis=0)
gen_best_score = sim_matrix.max(axis=0)

generated_match_rows = []

for j, gen_row in generated_df.iterrows():
    best_i = int(gen_best_idx[j])

    row = {
        "model": MODEL_LABEL,
        "generated_id": gen_row["generated_id"],
        "generated_intent_text": gen_row["generated_intent_text"],
        "best_RequirementID": reference_df.iloc[best_i]["RequirementID"],
        "best_TestCaseID": reference_df.iloc[best_i]["TestCaseID"],
        "best_reference_requirement": reference_df.iloc[best_i]["RequirementContent"],
        "best_reference_testcase": reference_df.iloc[best_i]["TestCaseContent"],
        "best_reference_intent_text": reference_df.iloc[best_i]["reference_intent_text"],
        "best_similarity": float(gen_best_score[j])
    }

    for tau in THRESHOLDS:
        row[f"generated_matched_at_{threshold_key(tau)}"] = int(gen_best_score[j] >= tau)

    generated_match_rows.append(row)

generated_matches_df = pd.DataFrame(generated_match_rows)

# =========================
# 10. STRICT ONE-TO-ONE MATCHING
# =========================
# Hungarian matching:
# Each reference test case can match at most one generated test case.
# Each generated test case can match at most one reference test case.

row_ind, col_ind = linear_sum_assignment(-sim_matrix)

one_to_one_rows = []

for r_idx, g_idx in zip(row_ind, col_ind):
    score = float(sim_matrix[r_idx, g_idx])

    row = {
        "model": MODEL_LABEL,
        "RequirementID": reference_df.iloc[r_idx]["RequirementID"],
        "TestCaseID": reference_df.iloc[r_idx]["TestCaseID"],
        "generated_id": generated_df.iloc[g_idx]["generated_id"],
        "one_to_one_similarity": score,
        "reference_intent_text": reference_df.iloc[r_idx]["reference_intent_text"],
        "generated_intent_text": generated_df.iloc[g_idx]["generated_intent_text"]
    }

    for tau in THRESHOLDS:
        row[f"one_to_one_matched_at_{threshold_key(tau)}"] = int(score >= tau)

    one_to_one_rows.append(row)

one_to_one_df = pd.DataFrame(one_to_one_rows)

# =========================
# 11. SUMMARY METRICS
# =========================

summary = {
    "model": MODEL_LABEL,
    "num_reference_cases": int(len(reference_df)),
    "num_generated_cases": int(len(generated_df)),
    "avg_reference_best_similarity": float(np.mean(ref_best_score)),
    "avg_generated_best_similarity": float(np.mean(gen_best_score)),
    "avg_one_to_one_similarity": float(one_to_one_df["one_to_one_similarity"].mean())
}

for tau in THRESHOLDS:
    key = threshold_key(tau)

    reference_recall = float(np.mean(ref_best_score >= tau))
    generated_precision = float(np.mean(gen_best_score >= tau))

    if reference_recall + generated_precision > 0:
        f1 = 2 * reference_recall * generated_precision / (reference_recall + generated_precision)
    else:
        f1 = 0.0

    one_to_one_matches = int(one_to_one_df[f"one_to_one_matched_at_{key}"].sum())

    summary[f"reference_recall_at_{key}"] = reference_recall
    summary[f"generated_precision_at_{key}"] = generated_precision
    summary[f"f1_at_{key}"] = float(f1)

    summary[f"one_to_one_matches_at_{key}"] = one_to_one_matches
    summary[f"one_to_one_recall_at_{key}"] = one_to_one_matches / len(reference_df)
    summary[f"one_to_one_precision_at_{key}"] = one_to_one_matches / len(generated_df)

summary_df = pd.DataFrame([summary])

# =========================
# 12. SAVE OUTPUTS
# =========================

summary_csv = os.path.join(OUTPUT_DIR, "intent_matching_summary.csv")
reference_matches_csv = os.path.join(OUTPUT_DIR, "reference_best_matches_intent.csv")
generated_matches_csv = os.path.join(OUTPUT_DIR, "generated_best_matches_intent.csv")
one_to_one_csv = os.path.join(OUTPUT_DIR, "strict_one_to_one_intent_matches.csv")
xlsx_path = os.path.join(OUTPUT_DIR, "external_requirement_intent_matching_results.xlsx")

summary_df.to_csv(summary_csv, index=False)
reference_matches_df.to_csv(reference_matches_csv, index=False)
generated_matches_df.to_csv(generated_matches_csv, index=False)
one_to_one_df.to_csv(one_to_one_csv, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    reference_matches_df.to_excel(writer, sheet_name="reference_best_matches", index=False)
    generated_matches_df.to_excel(writer, sheet_name="generated_best_matches", index=False)
    one_to_one_df.to_excel(writer, sheet_name="strict_one_to_one", index=False)
    reference_df.to_excel(writer, sheet_name="reference_intents", index=False)
    generated_df.to_excel(writer, sheet_name="generated_intents", index=False)

print("\nSaved outputs:")
print(summary_csv)
print(reference_matches_csv)
print(generated_matches_csv)
print(one_to_one_csv)
print(xlsx_path)

display(summary_df)

Using device: cuda
Loading model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded BGE-large using pytorch_model.bin route.
Model ready: BAAI/bge-large-en-v1.5
Reference cases: 30
Generated cases: 28
Saved intent text files:
/content/external_requirement_intent_matching_results/reference_intent_texts.csv
/content/external_requirement_intent_matching_results/generated_intent_texts.csv

Saved outputs:
/content/external_requirement_intent_matching_results/intent_matching_summary.csv
/content/external_requirement_intent_matching_results/reference_best_matches_intent.csv
/content/external_requirement_intent_matching_results/generated_best_matches_intent.csv
/content/external_requirement_intent_matching_results/strict_one_to_one_intent_matches.csv
/content/external_requirement_intent_matching_results/external_requirement_intent_matching_results.xlsx


,model,num_reference_cases,num_generated_cases,avg_reference_best_similarity,avg_generated_best_similarity,avg_one_to_one_similarity,reference_recall_at_0_65,generated_precision_at_0_65,f1_at_0_65,one_to_one_matches_at_0_65,...,f1_at_0_7,one_to_one_matches_at_0_7,one_to_one_recall_at_0_7,one_to_one_precision_at_0_7,reference_recall_at_0_75,generated_precision_at_0_75,f1_at_0_75,one_to_one_matches_at_0_75,one_to_one_recall_at_0_75,one_to_one_precision_at_0_75
0,Qwen-7B,30,28,0.875482,0.878185,0.876593,1.0,1.0,1.0,28,...,1.0,28,0.933333,1.0,1.0,1.0,1.0,28,0.933333,1.0


In [ ]:
!zip -r /content/external_requirement_intent_matching_results.zip /content/external_requirement_intent_matching_results

	zip warning: name not matched: /content/external_requirement_intent_matching_results

zip error: Nothing to do! (try: zip -r /content/external_requirement_intent_matching_results.zip . -i /content/external_requirement_intent_matching_results)


# last 30

In [ ]:
import pandas as pd

R_SET_PATH = "/content/R-set.csv"
T_SET_PATH = "/content/T-set.csv"

OUTPUT_DIR = "/content/external_requirement_benchmark_inputs"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

r_df = pd.read_csv(R_SET_PATH, sep=";", encoding="utf-8-sig", engine="python")
t_df = pd.read_csv(T_SET_PATH, sep=";", encoding="utf-8-sig", engine="python")

# UPDATED: take bottom/last 30 instead of top/first 30
r30 = r_df.tail(30).copy()
t30 = t_df.tail(30).copy()

r30.to_csv(os.path.join(OUTPUT_DIR, "R-set_last_30.csv"), index=False, sep=";")
t30.to_csv(os.path.join(OUTPUT_DIR, "T-set_last_30.csv"), index=False, sep=";")

print("Saved:")
print(os.path.join(OUTPUT_DIR, "R-set_last_30.csv"))
print(os.path.join(OUTPUT_DIR, "T-set_last_30.csv"))

print("\nSelected R-set range:")
print(r30[["RequirementID"]].head(1).to_string(index=False))
print(r30[["RequirementID"]].tail(1).to_string(index=False))

print("\nSelected T-set range:")
print(t30[["TestCaseID"]].head(1).to_string(index=False))
print(t30[["TestCaseID"]].tail(1).to_string(index=False))

Saved:
/content/external_requirement_benchmark_inputs/R-set_last_30.csv
/content/external_requirement_benchmark_inputs/T-set_last_30.csv

Selected R-set range:
RequirementID
        R-155
RequirementID
        R-184

Selected T-set range:
TestCaseID
    TC-155
TestCaseID
    TC-184


In [ ]:
# ============================================================
# External Requirement-to-Test Benchmark
# Intent-level semantic matching
#
# Input:
#   R-set_last_30.csv      -> last 30 requirements
#   T-set_last_30.csv      -> last 30 reference test cases
#   Geneated_testcases.csv -> generated test cases from one model
#
# Main idea:
#   Different formats are converted into a common "test intent" text.
#   Then BGE-large + cosine similarity is used for semantic matching.
#
# Outputs:
#   1. summary metrics
#   2. reference best matches
#   3. generated best matches
#   4. strict one-to-one matches
# ============================================================

import os
import re
import shutil
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

# =========================
# 1. FILE PATHS
# =========================

# UPDATED: now using last 30 files
R_SET_PATH = "/content/external_requirement_benchmark_inputs/R-set_last_30.csv"
T_SET_PATH = "/content/external_requirement_benchmark_inputs/T-set_last_30.csv"

# Keep this path same if your generated last-30 test cases file has this name.
# Change only if your bottom-30 generated output file has a different name.
GENERATED_PATH = "/content/Generated_testcases1.csv"

OUTPUT_DIR = "/content/external_requirement_intent_matching_results_last_30"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_LABEL = "Qwen-7B"   # change this for Qwen-14B, Mistral-7B, Mistral-14B

MODEL_NAME = "BAAI/bge-large-en-v1.5"
CACHE_DIR = "/content/bge_large_cache"

THRESHOLDS = [0.65, 0.70, 0.75]
PRIMARY_THRESHOLD = 0.70

USE_BGE_INSTRUCTION = True
BGE_INSTRUCTION = "Represent this software testing intent for semantic similarity matching: "

# Keep False if BGE-large is already downloaded.
CLEAN_LOCAL_MODEL_CACHE = False

# =========================
# 2. LOAD BGE-LARGE
# =========================

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

try:
    import torch
except Exception:
    torch = None

from sentence_transformers import SentenceTransformer

if CLEAN_LOCAL_MODEL_CACHE and os.path.exists(CACHE_DIR):
    print("Removing cache:", CACHE_DIR)
    shutil.rmtree(CACHE_DIR, ignore_errors=True)

os.makedirs(CACHE_DIR, exist_ok=True)

device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Loading model:", MODEL_NAME)

try:
    model = SentenceTransformer(
        MODEL_NAME,
        device=device,
        cache_folder=CACHE_DIR,
        model_kwargs={"use_safetensors": False}
    )
    print("Loaded BGE-large using pytorch_model.bin route.")
except TypeError:
    model = SentenceTransformer(
        MODEL_NAME,
        device=device,
        cache_folder=CACHE_DIR
    )
    print("Loaded BGE-large using standard route.")

print("Model ready:", MODEL_NAME)

# =========================
# 3. BASIC HELPERS
# =========================

def normalize_text(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = text.lower()
    text = text.replace("\n", " ")
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def threshold_key(t):
    return str(t).replace(".", "_")


def prepare_for_embedding(texts):
    cleaned = [normalize_text(t) for t in texts]
    if USE_BGE_INSTRUCTION:
        cleaned = [BGE_INSTRUCTION + t for t in cleaned]
    return cleaned


def encode_texts(texts, batch_size=16):
    return model.encode(
        prepare_for_embedding(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )


def get_first_existing_value(row, candidates):
    for c in candidates:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c]).strip()
    return ""


def read_semicolon_csv(path):
    return pd.read_csv(
        path,
        sep=";",
        encoding="utf-8-sig",
        engine="python"
    )


# =========================
# 4. REFERENCE TEST CASE PARSING
# =========================

def extract_section(text, start_keywords, end_keywords):
    raw = "" if text is None or pd.isna(text) else str(text)

    upper = raw.upper()
    start_pos = None

    for key in start_keywords:
        pos = upper.find(key.upper())
        if pos != -1:
            start_pos = pos + len(key)
            break

    if start_pos is None:
        return ""

    end_pos = len(raw)

    for end_key in end_keywords:
        pos = upper.find(end_key.upper(), start_pos)
        if pos != -1:
            end_pos = min(end_pos, pos)

    return raw[start_pos:end_pos].strip()


def remove_numbering(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = re.sub(r"\b\d+\.\s*", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_reference_intent(row):
    req_id = str(row.get("RequirementID", ""))
    requirement = str(row.get("RequirementContent", ""))
    test_case_id = str(row.get("TestCaseID", ""))
    test_content = str(row.get("TestCaseContent", ""))

    initial_conditions = extract_section(
        test_content,
        start_keywords=["INITIAL CONDITIONS"],
        end_keywords=["STEPS/DESCRIPTION", "EXPECTED RESULTS", "LINKS"]
    )

    steps = extract_section(
        test_content,
        start_keywords=["STEPS/DESCRIPTION", "STEPS"],
        end_keywords=["EXPECTED RESULTS", "LINKS"]
    )

    expected = extract_section(
        test_content,
        start_keywords=["EXPECTED RESULTS", "EXPECTED RESULT"],
        end_keywords=["LINKS"]
    )

    intent = (
        f"Requirement ID: {req_id}. "
        f"Requirement intent: {requirement}. "
        f"Reference test case ID: {test_case_id}. "
        f"Setup context: {remove_numbering(initial_conditions)}. "
        f"User action or test steps: {remove_numbering(steps)}. "
        f"Expected behavior: {remove_numbering(expected)}."
    )

    return intent


# =========================
# 5. GENERATED TEST CASE PARSING
# =========================

def build_generated_intent(row):
    generated_id = get_first_existing_value(
        row,
        ["Test Case ID", "test_case_id", "testcase_id", "ID", "id"]
    )

    title = get_first_existing_value(
        row,
        ["Title", "title", "Test Case Title", "test_title", "Name", "name"]
    )

    module = get_first_existing_value(
        row,
        ["Module", "module", "Feature", "feature", "Component", "component"]
    )

    requirement_id = get_first_existing_value(
        row,
        [
            "Requirement ID", "RequirementID", "requirement_id",
            "Source Requirement", "source_requirement", "Requirement"
        ]
    )

    preconditions = get_first_existing_value(
        row,
        ["Preconditions", "preconditions", "Precondition", "precondition"]
    )

    test_data = get_first_existing_value(
        row,
        ["Test Data", "test_data", "Data", "data"]
    )

    steps = get_first_existing_value(
        row,
        ["Test Steps", "test_steps", "Steps", "steps", "Procedure", "procedure"]
    )

    expected = get_first_existing_value(
        row,
        ["Expected Result", "expected_result", "Expected Results", "expected_results", "Oracle", "oracle"]
    )

    all_columns_text = " ".join(
        [str(x) for x in row.values if pd.notna(x) and str(x).strip()]
    )

    intent = (
        f"Generated test case ID: {generated_id}. "
        f"Test objective title: {title}. "
        f"Feature or module: {module}. "
        f"Requirement reference: {requirement_id}. "
        f"Precondition context: {preconditions}. "
        f"Input or test data: {test_data}. "
        f"User action or test steps: {remove_numbering(steps)}. "
        f"Expected behavior: {remove_numbering(expected)}. "
        f"Full generated case: {all_columns_text}."
    )

    return intent


def read_generated_cases(path):
    df = pd.read_csv(path).copy().reset_index(drop=True)

    if "Test Case ID" in df.columns:
        df["generated_id"] = df["Test Case ID"].astype(str)
    elif "test_case_id" in df.columns:
        df["generated_id"] = df["test_case_id"].astype(str)
    elif "testcase_id" in df.columns:
        df["generated_id"] = df["testcase_id"].astype(str)
    elif "ID" in df.columns:
        df["generated_id"] = df["ID"].astype(str)
    else:
        df["generated_id"] = [f"GEN-{i+1:03d}" for i in range(len(df))]

    df["generated_intent_text"] = df.apply(build_generated_intent, axis=1)
    df["generated_intent_norm"] = df["generated_intent_text"].apply(normalize_text)

    return df


# =========================
# 6. LOAD R-SET, T-SET, GENERATED CASES
# =========================

r_df = read_semicolon_csv(R_SET_PATH)
t_df = read_semicolon_csv(T_SET_PATH)

# UPDATED: these files already contain the last 30 requirement-test pairs
r30 = r_df.copy().reset_index(drop=True)
t30 = t_df.copy().reset_index(drop=True)

reference_df = pd.concat(
    [
        r30[["RequirementID", "RequirementContent"]],
        t30[["TestCaseID", "TestCaseContent"]]
    ],
    axis=1
)

reference_df["reference_intent_text"] = reference_df.apply(build_reference_intent, axis=1)
reference_df["reference_intent_norm"] = reference_df["reference_intent_text"].apply(normalize_text)

generated_df = read_generated_cases(GENERATED_PATH)

print("Reference cases:", len(reference_df))
print("Generated cases:", len(generated_df))

print("\nReference requirement range:")
print(reference_df[["RequirementID", "TestCaseID"]].head(1).to_string(index=False))
print(reference_df[["RequirementID", "TestCaseID"]].tail(1).to_string(index=False))

reference_intent_path = os.path.join(OUTPUT_DIR, "reference_intent_texts.csv")
generated_intent_path = os.path.join(OUTPUT_DIR, "generated_intent_texts.csv")

reference_df.to_csv(reference_intent_path, index=False)
generated_df.to_csv(generated_intent_path, index=False)

print("Saved intent text files:")
print(reference_intent_path)
print(generated_intent_path)

# =========================
# 7. INTENT-LEVEL SEMANTIC MATCHING
# =========================

ref_emb = encode_texts(reference_df["reference_intent_norm"].tolist())
gen_emb = encode_texts(generated_df["generated_intent_norm"].tolist())

sim_matrix = cosine_similarity(ref_emb, gen_emb)

# =========================
# 8. REFERENCE RECALL SIDE
# =========================

ref_best_idx = sim_matrix.argmax(axis=1)
ref_best_score = sim_matrix.max(axis=1)

reference_match_rows = []

for i, ref_row in reference_df.iterrows():
    best_j = int(ref_best_idx[i])

    row = {
        "model": MODEL_LABEL,
        "RequirementID": ref_row["RequirementID"],
        "TestCaseID": ref_row["TestCaseID"],
        "RequirementContent": ref_row["RequirementContent"],
        "ReferenceTestCaseContent": ref_row["TestCaseContent"],
        "reference_intent_text": ref_row["reference_intent_text"],
        "best_generated_id": generated_df.iloc[best_j]["generated_id"],
        "best_generated_intent_text": generated_df.iloc[best_j]["generated_intent_text"],
        "best_similarity": float(ref_best_score[i])
    }

    for tau in THRESHOLDS:
        row[f"reference_matched_at_{threshold_key(tau)}"] = int(ref_best_score[i] >= tau)

    reference_match_rows.append(row)

reference_matches_df = pd.DataFrame(reference_match_rows)

# =========================
# 9. GENERATED PRECISION SIDE
# =========================

gen_best_idx = sim_matrix.argmax(axis=0)
gen_best_score = sim_matrix.max(axis=0)

generated_match_rows = []

for j, gen_row in generated_df.iterrows():
    best_i = int(gen_best_idx[j])

    row = {
        "model": MODEL_LABEL,
        "generated_id": gen_row["generated_id"],
        "generated_intent_text": gen_row["generated_intent_text"],
        "best_RequirementID": reference_df.iloc[best_i]["RequirementID"],
        "best_TestCaseID": reference_df.iloc[best_i]["TestCaseID"],
        "best_reference_requirement": reference_df.iloc[best_i]["RequirementContent"],
        "best_reference_testcase": reference_df.iloc[best_i]["TestCaseContent"],
        "best_reference_intent_text": reference_df.iloc[best_i]["reference_intent_text"],
        "best_similarity": float(gen_best_score[j])
    }

    for tau in THRESHOLDS:
        row[f"generated_matched_at_{threshold_key(tau)}"] = int(gen_best_score[j] >= tau)

    generated_match_rows.append(row)

generated_matches_df = pd.DataFrame(generated_match_rows)

# =========================
# 10. STRICT ONE-TO-ONE MATCHING
# =========================

row_ind, col_ind = linear_sum_assignment(-sim_matrix)

one_to_one_rows = []

for r_idx, g_idx in zip(row_ind, col_ind):
    score = float(sim_matrix[r_idx, g_idx])

    row = {
        "model": MODEL_LABEL,
        "RequirementID": reference_df.iloc[r_idx]["RequirementID"],
        "TestCaseID": reference_df.iloc[r_idx]["TestCaseID"],
        "generated_id": generated_df.iloc[g_idx]["generated_id"],
        "one_to_one_similarity": score,
        "reference_intent_text": reference_df.iloc[r_idx]["reference_intent_text"],
        "generated_intent_text": generated_df.iloc[g_idx]["generated_intent_text"]
    }

    for tau in THRESHOLDS:
        row[f"one_to_one_matched_at_{threshold_key(tau)}"] = int(score >= tau)

    one_to_one_rows.append(row)

one_to_one_df = pd.DataFrame(one_to_one_rows)

# =========================
# 11. SUMMARY METRICS
# =========================

summary = {
    "model": MODEL_LABEL,
    "num_reference_cases": int(len(reference_df)),
    "num_generated_cases": int(len(generated_df)),
    "avg_reference_best_similarity": float(np.mean(ref_best_score)),
    "avg_generated_best_similarity": float(np.mean(gen_best_score)),
    "avg_one_to_one_similarity": float(one_to_one_df["one_to_one_similarity"].mean())
}

for tau in THRESHOLDS:
    key = threshold_key(tau)

    reference_recall = float(np.mean(ref_best_score >= tau))
    generated_precision = float(np.mean(gen_best_score >= tau))

    if reference_recall + generated_precision > 0:
        f1 = 2 * reference_recall * generated_precision / (reference_recall + generated_precision)
    else:
        f1 = 0.0

    one_to_one_matches = int(one_to_one_df[f"one_to_one_matched_at_{key}"].sum())

    summary[f"reference_recall_at_{key}"] = reference_recall
    summary[f"generated_precision_at_{key}"] = generated_precision
    summary[f"f1_at_{key}"] = float(f1)

    summary[f"one_to_one_matches_at_{key}"] = one_to_one_matches
    summary[f"one_to_one_recall_at_{key}"] = one_to_one_matches / len(reference_df)
    summary[f"one_to_one_precision_at_{key}"] = one_to_one_matches / len(generated_df)

summary_df = pd.DataFrame([summary])

# =========================
# 12. SAVE OUTPUTS
# =========================

summary_csv = os.path.join(OUTPUT_DIR, "intent_matching_summary.csv")
reference_matches_csv = os.path.join(OUTPUT_DIR, "reference_best_matches_intent.csv")
generated_matches_csv = os.path.join(OUTPUT_DIR, "generated_best_matches_intent.csv")
one_to_one_csv = os.path.join(OUTPUT_DIR, "strict_one_to_one_intent_matches.csv")
xlsx_path = os.path.join(OUTPUT_DIR, "external_requirement_intent_matching_results_last_30.xlsx")

summary_df.to_csv(summary_csv, index=False)
reference_matches_df.to_csv(reference_matches_csv, index=False)
generated_matches_df.to_csv(generated_matches_csv, index=False)
one_to_one_df.to_csv(one_to_one_csv, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    reference_matches_df.to_excel(writer, sheet_name="reference_best_matches", index=False)
    generated_matches_df.to_excel(writer, sheet_name="generated_best_matches", index=False)
    one_to_one_df.to_excel(writer, sheet_name="strict_one_to_one", index=False)
    reference_df.to_excel(writer, sheet_name="reference_intents", index=False)
    generated_df.to_excel(writer, sheet_name="generated_intents", index=False)

print("\nSaved outputs:")
print(summary_csv)
print(reference_matches_csv)
print(generated_matches_csv)
print(one_to_one_csv)
print(xlsx_path)

display(summary_df)

Using device: cuda
Loading model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded BGE-large using pytorch_model.bin route.
Model ready: BAAI/bge-large-en-v1.5
Reference cases: 30
Generated cases: 27

Reference requirement range:
RequirementID TestCaseID
        R-155     TC-155
RequirementID TestCaseID
        R-184     TC-184
Saved intent text files:
/content/external_requirement_intent_matching_results_last_30/reference_intent_texts.csv
/content/external_requirement_intent_matching_results_last_30/generated_intent_texts.csv

Saved outputs:
/content/external_requirement_intent_matching_results_last_30/intent_matching_summary.csv
/content/external_requirement_intent_matching_results_last_30/reference_best_matches_intent.csv
/content/external_requirement_intent_matching_results_last_30/generated_best_matches_intent.csv
/content/external_requirement_intent_matching_results_last_30/strict_one_to_one_intent_matches.csv
/content/external_requirement_intent_matching_results_last_30/external_requirement_intent_matching_results_last_30.xlsx


,model,num_reference_cases,num_generated_cases,avg_reference_best_similarity,avg_generated_best_similarity,avg_one_to_one_similarity,reference_recall_at_0_65,generated_precision_at_0_65,f1_at_0_65,one_to_one_matches_at_0_65,...,f1_at_0_7,one_to_one_matches_at_0_7,one_to_one_recall_at_0_7,one_to_one_precision_at_0_7,reference_recall_at_0_75,generated_precision_at_0_75,f1_at_0_75,one_to_one_matches_at_0_75,one_to_one_recall_at_0_75,one_to_one_precision_at_0_75
0,Qwen-7B,30,27,0.88159,0.887342,0.885083,1.0,1.0,1.0,27,...,1.0,27,0.9,1.0,1.0,1.0,1.0,27,0.9,1.0


In [ ]:
!zip -r /content/external_requirement_intent_matching_results_last_30.zip /content/external_requirement_intent_matching_results_last_30

  adding: content/external_requirement_intent_matching_results_last_30/ (stored 0%)
  adding: content/external_requirement_intent_matching_results_last_30/intent_matching_summary.csv (deflated 69%)
  adding: content/external_requirement_intent_matching_results_last_30/strict_one_to_one_intent_matches.csv (deflated 85%)
  adding: content/external_requirement_intent_matching_results_last_30/reference_best_matches_intent.csv (deflated 89%)
  adding: content/external_requirement_intent_matching_results_last_30/generated_best_matches_intent.csv (deflated 89%)
  adding: content/external_requirement_intent_matching_results_last_30/reference_intent_texts.csv (deflated 89%)
  adding: content/external_requirement_intent_matching_results_last_30/generated_intent_texts.csv (deflated 91%)
  adding: content/external_requirement_intent_matching_results_last_30/external_requirement_intent_matching_results_last_30.xlsx (deflated 2%)


# EBT

In [ ]:
# ============================================================
# EBT dataset setup
# Creates requirement input + held-out test reference files
# ============================================================

!pip -q install datasets pandas reportlab pyarrow

import os
import re
import itertools
import shutil
import pandas as pd
from datasets import load_dataset, get_dataset_config_names

DATASET_ID = "thearod5/ebt"
OUT_DIR = "/content/ebt_requirement_test_setup"
os.makedirs(OUT_DIR, exist_ok=True)

print("Available configs:")
configs = get_dataset_config_names(DATASET_ID)
print(configs)

def load_hf_config(config_name):
    ds = load_dataset(DATASET_ID, config_name, split="train")
    return ds.to_pandas()

artifacts_df = load_hf_config("artifacts")

try:
    traces_df = load_hf_config("traces")
except Exception as e:
    print("Could not load traces config:", e)
    traces_df = pd.DataFrame()

try:
    matrices_df = load_hf_config("matrices")
except Exception as e:
    print("Could not load matrices config:", e)
    matrices_df = pd.DataFrame()

print("\nArtifacts shape:", artifacts_df.shape)
print("Artifacts columns:", artifacts_df.columns.tolist())

print("\nLayer counts:")
print(artifacts_df["layer"].value_counts(dropna=False))

print("\nTraces shape:", traces_df.shape)
print("Traces columns:", traces_df.columns.tolist())

print("\nMatrices shape:", matrices_df.shape)
print("Matrices columns:", matrices_df.columns.tolist())

display(artifacts_df.head())
display(traces_df.head())
display(matrices_df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.1 MB/s eta 0:00:00
Available configs:


README.md:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

['artifacts', 'traces', 'matrices', 'train']


artifacts.csv:   0%|          | 0.00/79.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/116 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/2 [00:00<?, ? examples/s]


Artifacts shape: (116, 4)
Artifacts columns: ['id', 'content', 'layer', 'summary']

Layer counts:
layer
Code           50
Requirement    41
Test           25
Name: count, dtype: int64

Traces shape: (144, 3)
Traces columns: ['s_id', 't_id', 'label']

Matrices shape: (2, 2)
Matrices columns: ['source_type', 'target_type']


,id,content,layer,summary
0,100,Requirements shall be managed in a 3rd party r...,Requirement,NaN
1,101,Non-requirement artifacts shall be managed in ...,Requirement,NaN
2,102,Only registered subscribers shall be allowed t...,Requirement,NaN
3,103,A user shall register as a subscriber.,Requirement,NaN
4,104,On registration a subscriber shall register it...,Requirement,NaN


,s_id,t_id,label
0,141,102,1
1,143,103,1
2,143,104,1
3,144,104,1
4,141,105,1


,source_type,target_type
0,Test,Requirement
1,Code,Test


In [ ]:
# ============================================================
# Split EBT artifacts into requirements and test cases
# ============================================================

def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def norm_id(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s.split(".")[0]
    return s

artifacts = artifacts_df.copy()
artifacts.columns = [c.strip().lower() for c in artifacts.columns]

id_col = "id"
content_col = "content"
layer_col = "layer"

artifacts["artifact_id"] = artifacts[id_col].apply(norm_id)
artifacts["content_clean"] = artifacts[content_col].apply(clean_text)
artifacts["layer_clean"] = artifacts[layer_col].apply(clean_text)

requirements_df = artifacts[
    artifacts["layer_clean"].str.lower().eq("requirement")
].copy()

testcases_df = artifacts[
    artifacts["layer_clean"].str.lower().eq("test")
].copy()

requirements_df = requirements_df.reset_index(drop=True)
testcases_df = testcases_df.reset_index(drop=True)

print("Requirements:", len(requirements_df))
print("Test cases:", len(testcases_df))

print("\nRequirement sample:")
display(requirements_df[["artifact_id", "layer_clean", "content_clean"]].head(10))

print("\nTest-case sample:")
display(testcases_df[["artifact_id", "layer_clean", "content_clean"]].head(10))

Requirements: 41
Test cases: 25

Requirement sample:


,artifact_id,layer_clean,content_clean
0,100,Requirement,Requirements shall be managed in a 3rd party r...
1,101,Requirement,Non-requirement artifacts shall be managed in ...
2,102,Requirement,Only registered subscribers shall be allowed t...
3,103,Requirement,A user shall register as a subscriber.
4,104,Requirement,On registration a subscriber shall register it...
5,105,Requirement,The user shall establish traces between requir...
6,106,Requirement,Each artifact shall be placed under the contro...
7,107,Requirement,The user shall be able to delete subscriptions.
8,108,Requirement,A subscribermanager shall register itself with...
9,109,Requirement,If the subscribermanager is online; the push m...



Test-case sample:


,artifact_id,layer_clean,content_clean
0,141,Test,Test case: Establish Trace (2.1.1)(2.2.1) Prec...
1,142,Test,Test case: Non-registered subscriber attempts ...
2,143,Test,Test case: Subscriber registers with subscribe...
3,144,Test,Test case: Subscriber fails to register with s...
4,145,Test,Test case: Delete subscriptions (2.2.3) Precon...
5,146,Test,Test case: Subscriber Manager registers with e...
6,147,Test,Test case: An event is published in real time ...
7,148,Test,Test case: Subscriber Manager comes online and...
8,149,Test,Test case: Subscriber manager receives non-spe...
9,150,Test,Test case: Subscriber manager receives specula...


In [ ]:
# ============================================================
# Extract requirement-to-test trace links
# This is robust to different column names in the traces file.
# ============================================================

req_ids = set(requirements_df["artifact_id"].astype(str))
test_ids = set(testcases_df["artifact_id"].astype(str))

def extract_req_test_links(traces_df, req_ids, test_ids):
    if traces_df is None or traces_df.empty:
        return pd.DataFrame(columns=["requirement_id", "testcase_id"])

    temp = traces_df.copy()
    temp.columns = [c.strip().lower() for c in temp.columns]

    # Normalize all cells to comparable IDs
    for c in temp.columns:
        temp[c] = temp[c].apply(norm_id)

    best_pairs = []
    best_count = 0
    best_cols = None

    cols = list(temp.columns)

    for c1, c2 in itertools.combinations(cols, 2):
        pairs = []

        for _, row in temp.iterrows():
            a = str(row[c1]).strip()
            b = str(row[c2]).strip()

            if a in req_ids and b in test_ids:
                pairs.append((a, b))
            elif b in req_ids and a in test_ids:
                pairs.append((b, a))

        unique_pairs = sorted(set(pairs))

        if len(unique_pairs) > best_count:
            best_count = len(unique_pairs)
            best_pairs = unique_pairs
            best_cols = (c1, c2)

    print("Best trace columns:", best_cols)
    print("Requirement-test links found:", best_count)

    return pd.DataFrame(best_pairs, columns=["requirement_id", "testcase_id"])

req_test_links_df = extract_req_test_links(traces_df, req_ids, test_ids)

if len(req_test_links_df) == 0:
    print("\nWARNING: No requirement-test links found from traces.")
    print("The code will use all requirements and all test cases.")
    linked_requirements_df = requirements_df.copy()
    linked_testcases_df = testcases_df.copy()
else:
    linked_req_ids = set(req_test_links_df["requirement_id"].astype(str))
    linked_test_ids = set(req_test_links_df["testcase_id"].astype(str))

    linked_requirements_df = requirements_df[
        requirements_df["artifact_id"].isin(linked_req_ids)
    ].copy().reset_index(drop=True)

    linked_testcases_df = testcases_df[
        testcases_df["artifact_id"].isin(linked_test_ids)
    ].copy().reset_index(drop=True)

print("\nLinked requirements:", len(linked_requirements_df))
print("Linked test cases:", len(linked_testcases_df))

display(req_test_links_df.head(20))
display(linked_requirements_df[["artifact_id", "content_clean"]].head())
display(linked_testcases_df[["artifact_id", "content_clean"]].head())

Best trace columns: ('s_id', 't_id')
Requirement-test links found: 51

Linked requirements: 39
Linked test cases: 25


,requirement_id,testcase_id
0,102,141
1,103,143
2,104,143
3,104,144
4,105,141
5,105,142
6,106,143
7,106,144
8,107,145
9,108,146


,artifact_id,content_clean
0,102,Only registered subscribers shall be allowed t...
1,103,A user shall register as a subscriber.
2,104,On registration a subscriber shall register it...
3,105,The user shall establish traces between requir...
4,106,Each artifact shall be placed under the contro...


,artifact_id,content_clean
0,141,Test case: Establish Trace (2.1.1)(2.2.1) Prec...
1,142,Test case: Non-registered subscriber attempts ...
2,143,Test case: Subscriber registers with subscribe...
3,144,Test case: Subscriber fails to register with s...
4,145,Test case: Delete subscriptions (2.2.3) Precon...


In [ ]:
# ============================================================
# Create EBT requirement PDF/TXT and held-out test-case CSV/PDF
# ============================================================

from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

REQ_PDF = os.path.join(OUT_DIR, "ebt_requirements_linked_only.pdf")
REQ_TXT = os.path.join(OUT_DIR, "ebt_requirements_linked_only.txt")
REQ_CSV = os.path.join(OUT_DIR, "ebt_requirements_linked_only.csv")

TEST_PDF = os.path.join(OUT_DIR, "ebt_reference_testcases_linked_only.pdf")
TEST_CSV = os.path.join(OUT_DIR, "ebt_reference_testcases_linked_only.csv")

MAPPING_CSV = os.path.join(OUT_DIR, "ebt_requirement_test_links.csv")

def safe_html(x):
    return escape("" if pd.isna(x) else str(x))

def soft_break_testcase(text):
    text = clean_text(text)
    markers = [
        "Test case:",
        "Preconditions",
        "Preconditions:",
        "Steps",
        "Steps:",
        "Postconditions",
        "Postconditions:",
        "Post conditions",
        "Post conditions:",
    ]

    for m in markers:
        text = text.replace(" " + m, "\n" + m)

    text = re.sub(r"\n{2,}", "\n", text).strip()
    return text

def create_pdf(df, output_path, title, subtitle, id_label, content_label, is_test=False):
    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=0.65 * inch,
        leftMargin=0.65 * inch,
        topMargin=0.65 * inch,
        bottomMargin=0.65 * inch,
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "TitleStyle",
        parent=styles["Title"],
        fontSize=18,
        leading=22,
        alignment=TA_LEFT,
        spaceAfter=12,
    )

    subtitle_style = ParagraphStyle(
        "SubtitleStyle",
        parent=styles["BodyText"],
        fontSize=9,
        leading=12,
        textColor=colors.darkgray,
        spaceAfter=12,
    )

    heading_style = ParagraphStyle(
        "HeadingStyle",
        parent=styles["Heading2"],
        fontSize=12,
        leading=15,
        spaceBefore=10,
        spaceAfter=6,
    )

    body_style = ParagraphStyle(
        "BodyStyle",
        parent=styles["BodyText"],
        fontSize=9,
        leading=12,
        spaceAfter=8,
    )

    meta_style = ParagraphStyle(
        "MetaStyle",
        parent=styles["BodyText"],
        fontSize=8,
        leading=10,
        textColor=colors.darkgray,
        spaceAfter=4,
    )

    story = []

    story.append(Paragraph(safe_html(title), title_style))
    story.append(Paragraph(safe_html(subtitle), subtitle_style))

    table_data = [["#", id_label, "Layer"]]
    for idx, row in df.iterrows():
        table_data.append([
            str(idx + 1),
            str(row["artifact_id"]),
            str(row["layer_clean"]),
        ])

    table = Table(table_data, colWidths=[0.35 * inch, 1.3 * inch, 3.6 * inch])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))

    story.append(table)
    story.append(Spacer(1, 12))

    for idx, row in df.iterrows():
        artifact_id = str(row["artifact_id"])
        layer = str(row["layer_clean"])
        content = row["content_clean"]

        if is_test:
            content = soft_break_testcase(content)

        story.append(Paragraph(f"{idx + 1}. {safe_html(id_label)} {safe_html(artifact_id)}", heading_style))
        story.append(Paragraph(f"<b>{safe_html(id_label)}:</b> {safe_html(artifact_id)}", meta_style))
        story.append(Paragraph(f"<b>Layer:</b> {safe_html(layer)}", meta_style))
        story.append(Paragraph(
            f"<b>{safe_html(content_label)}:</b><br/>{safe_html(content).replace(chr(10), '<br/>')}",
            body_style
        ))
        story.append(Spacer(1, 8))

    doc.build(story)

# Save requirement CSV
req_export = linked_requirements_df[["artifact_id", "layer_clean", "content_clean"]].copy()
req_export.rename(columns={
    "artifact_id": "RequirementID",
    "layer_clean": "Layer",
    "content_clean": "RequirementContent"
}, inplace=True)
req_export.to_csv(REQ_CSV, index=False)

# Save requirement TXT
with open(REQ_TXT, "w", encoding="utf-8") as f:
    f.write("EBT Linked Requirements Input\n")
    f.write("This file is for LLM input only. Reference test cases are not included.\n\n")

    for _, row in linked_requirements_df.iterrows():
        f.write("-" * 80 + "\n")
        f.write(f"RequirementID: {row['artifact_id']}\n")
        f.write(f"Layer: {row['layer_clean']}\n")
        f.write(f"RequirementContent: {row['content_clean']}\n\n")

# Save test CSV
test_export = linked_testcases_df[["artifact_id", "layer_clean", "content_clean"]].copy()
test_export.rename(columns={
    "artifact_id": "TestCaseID",
    "layer_clean": "Layer",
    "content_clean": "TestCaseContent"
}, inplace=True)
test_export.to_csv(TEST_CSV, index=False)

# Save mapping CSV
req_test_links_df.to_csv(MAPPING_CSV, index=False)

# Create PDFs
create_pdf(
    linked_requirements_df,
    REQ_PDF,
    title="EBT Linked Requirements Input",
    subtitle="Input-only requirement artifact generated from the EBT traceability dataset. Reference test cases are not included.",
    id_label="RequirementID",
    content_label="Requirement content",
    is_test=False
)

create_pdf(
    linked_testcases_df,
    TEST_PDF,
    title="EBT Held-Out Reference Test Cases",
    subtitle="Held-out reference test cases from the EBT traceability dataset. This file should not be given to the LLM during generation.",
    id_label="TestCaseID",
    content_label="Test-case content",
    is_test=True
)

print("Created files:")
print("Requirement PDF:", REQ_PDF)
print("Requirement TXT:", REQ_TXT)
print("Requirement CSV:", REQ_CSV)
print("Test-case PDF:", TEST_PDF)
print("Test-case CSV:", TEST_CSV)
print("Mapping CSV:", MAPPING_CSV)

print("\nCounts:")
print("Linked requirements:", len(linked_requirements_df))
print("Linked test cases:", len(linked_testcases_df))
print("Requirement-test links:", len(req_test_links_df))

Created files:
Requirement PDF: /content/ebt_requirement_test_setup/ebt_requirements_linked_only.pdf
Requirement TXT: /content/ebt_requirement_test_setup/ebt_requirements_linked_only.txt
Requirement CSV: /content/ebt_requirement_test_setup/ebt_requirements_linked_only.csv
Test-case PDF: /content/ebt_requirement_test_setup/ebt_reference_testcases_linked_only.pdf
Test-case CSV: /content/ebt_requirement_test_setup/ebt_reference_testcases_linked_only.csv
Mapping CSV: /content/ebt_requirement_test_setup/ebt_requirement_test_links.csv

Counts:
Linked requirements: 39
Linked test cases: 25
Requirement-test links: 51


In [ ]:
# ============================================================
# Zip output files
# ============================================================

ZIP_PATH = "/content/ebt_requirement_test_setup.zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive(
    ZIP_PATH.replace(".zip", ""),
    "zip",
    OUT_DIR
)

print("Download ZIP:")
print(ZIP_PATH)

Download ZIP:
/content/ebt_requirement_test_setup.zip
